# 11 — Static Validation (Synthetic Expert Data)

**Question:** Can GS-CV identify BE vs SC
and recover parameters from expert-phase data with known ground truth?

**Protocol:**
1. Quick run:
	- Generate synthetic cohort: N BE + N SC animals, static expert params,uniform distribution, 15 sessions × 350 trials each
	- Run GS-CV on each (model selection + parameter recovery)
	- Analyse the results: accuracy, recovery correlation, confusion matrices
2. Full run:
	- Load the cluster validation results
	- Analyse the results

All ground truth is known — this is a pure methods validation.

In [ ]:
%matplotlib inline
from shared_setup import *

# Model-identification / validation infrastructure
from scripts.config import cohort_path, results_dir, BASE_SEED, GS_BURN_IN
from scripts.providers import load_animals
from scripts.run_gs import run_gs_cohort
from scripts.validation.generate_synthetic_cohort import generate_cohort
from utils.cv_utils import load_cv_results
from plotting.cv import plot_cv_comparison, plot_confusion, plot_recovery


## 1. Quick (local) Run

### 1a. Configuration

In [ ]:
# ── Configuration ────────────────────────────────────────
COHORT     = 'static_uniform'
FIT_TARGET = 'update_matrix'

# Quick-run scale (coarse 384-point grid; a few minutes of compute per
# animal/model on a multi-core machine). Dial these down for faster iteration.
QUICK_COHORT      = f'{COHORT}_quick'
QUICK_N_PER_MODEL = 2
QUICK_N_SEEDS     = 2
QUICK_N_SESSIONS  = 6
QUICK_N_TRIALS    = 250

QUICK_DIR = results_dir('grid_search', 'quick', COHORT, FIT_TARGET)
print('quick results ->', QUICK_DIR)


# ── Display helpers (shared by the quick and full sections) ──────────────
def show_model_id(res, label=''):
    """Per-animal CV comparison plots, then the cohort confusion matrix."""
    for aid in res.comparison['animal_id']:
        plot_cv_comparison(res.long, res.comparison, aid, fit_target=FT_LABEL[FIT_TARGET])
        plt.show()
    fig, ax = plt.subplots(figsize=(4, 4))
    plot_confusion(res.comparison, ax=ax)
    ax.set_title(f'{label} {ax.get_title()}'.strip())
    plt.show()


def show_recovery(res, label=''):
    """Grid of true-vs-recovered scatter, one panel per parameter."""
    params = list(res.recovery['param'].unique())
    if not params:
        print('No recovery data.')
        return
    ncol = 4
    nrow = int(np.ceil(len(params) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3.5 * nrow))
    axes = np.atleast_1d(axes).ravel()
    for ax, param in zip(axes, params):
        plot_recovery(res.recovery, param, ax=ax)
    for ax in axes[len(params):]:
        ax.set_visible(False)
    fig.suptitle(f'{label} parameter recovery (true-model fit)'.strip(), fontsize=12)
    fig.tight_layout()
    plt.show()


### 1b. Generate synthetic cohort

In [ ]:
# Generate a small expert cohort for the quick run, save it, then load as records.
# (The full run uses the larger cohort generated on the cluster - see section 2.)
import pickle

cohort_path(QUICK_COHORT).parent.mkdir(parents=True, exist_ok=True)
animals = generate_cohort(
    n_per_model=QUICK_N_PER_MODEL, n_sessions=QUICK_N_SESSIONS,
    n_trials=QUICK_N_TRIALS, burn_in=GS_BURN_IN,
    min_accuracy=0.65, max_attempts=200, seed=BASE_SEED,
)
with open(cohort_path(QUICK_COHORT), 'wb') as f:
    pickle.dump({'cohort': QUICK_COHORT, 'animals': animals}, f)

records = load_animals('synthetic', QUICK_COHORT)
print(f'{len(records)} animals:', [(r.animal_id, r.true_model) for r in records])


### 1c. Grid-search Results

#### 1ci. Grid-search model identification

In [ ]:
# Coarse grid-search CV per (animal, model) over QUICK_N_SEEDS seeds.
# Slow cell: ~a few minutes per (animal, model) on a multi-core machine.
paths = run_gs_cohort(records, QUICK_DIR, n_seeds=QUICK_N_SEEDS,
                      fit_target=FIT_TARGET, coarse=True)
print(f'\nwrote {len(paths)} result pickles to {QUICK_DIR}')


In [ ]:
res = load_cv_results(QUICK_DIR)
show_model_id(res, label='Quick:')


#### 1cii. GS parameter recovery

For correctly identified animals, compare the best-fit parameters
(from the best grid point) against the true generating parameters.

In [ ]:
show_recovery(res, label='Quick:')


### 1d. Diagnostic — inspect one animal

Two checks on a single animal (set `DIAG_ANIMAL`), to tell apart a coarse-grid
artefact from genuine BE/SC non-identifiability:

1. **Grid resolution.** Re-fit the *true* model with a one-point grid pinned at
   the true parameters, through the same CV pipeline. If `true@true-params` is far
   below `true@coarse-grid`, the coarse grid is missing good params. If they are
   comparable and the *other* model still wins at the coarse grid, it is genuine
   non-identifiability under this metric, not the grid.
2. **Update matrix + psychometric** of the data against the best-fit BE and SC,
   to see *where* each model matches or fails (sequential structure vs static curve).


In [ ]:
import pickle
from behav_utils.data.synthetic import session_from_arrays
from models.simulate import simulate_choices
from analysis.grid_search import ParameterGrid, compute_grid_search_cv
from scripts.config import GS_N_FOLDS, GS_N_BINS
from utils.cv_utils import compute_seed_errors
from behav_utils.analysis.update_matrix import matrix_error

DIAG_ANIMAL = 'BE00'

diag = next(a for a in animals if a['animal_id'] == DIAG_ANIMAL)   # raw cohort arrays
rec  = next(r for r in records if r.animal_id == DIAG_ANIMAL)      # reconstructed sessions

be_err, be_best = compute_seed_errors(pickle.load(open(QUICK_DIR / f'{DIAG_ANIMAL}_BE.pkl', 'rb')))
sc_err, sc_best = compute_seed_errors(pickle.load(open(QUICK_DIR / f'{DIAG_ANIMAL}_SC.pkl', 'rb')))
coarse = {'BE': float(np.mean(be_err)), 'SC': float(np.mean(sc_err))}


def single_point_grid(true_model, tp):
    """One-point ParameterGrid pinned at the true parameters."""
    p1, p2 = ('eta_learning', 'eta_relax') if true_model == 'BE' else ('gamma', 'sigma_update')
    return ParameterGrid(
        np.array([tp['sigma_percep']]), np.array([tp['A_repulsion']]),
        np.array([tp[p1]]), np.array([tp[p2]]), p1, p2,
    )


# Diagnostic 1: error of the TRUE model at its TRUE params (identical CV pipeline).
r_true = compute_grid_search_cv(
    rec.sessions, rec.true_model,
    grid=single_point_grid(rec.true_model, rec.true_params),
    n_folds=GS_N_FOLDS, seed=BASE_SEED + 1, burn_in=GS_BURN_IN,
    n_bins=GS_N_BINS, fit_target=FIT_TARGET,
)
tm, other = rec.true_model, ('SC' if rec.true_model == 'BE' else 'BE')
print(f'{DIAG_ANIMAL} (true {tm}):')
print(f'  {tm}@coarse-grid = {coarse[tm]:.4f}')
print(f'  {tm}@true-params = {r_true["avg_test_error"]:.4f}')
print(f'  {other}@coarse-grid = {coarse[other]:.4f}\n')
if r_true['avg_test_error'] < coarse[tm] - 1e-4:
    print(f'  -> coarse grid leaves error on the table for {tm}.')
if coarse[other] < coarse[tm]:
    flips = r_true['avg_test_error'] < coarse[other]
    print(f'  -> {other} wins at coarse grid; {tm}@true-params would '
          + ('WIN -> looks like a grid artefact.' if flips
             else 'still lose -> looks like genuine non-identifiability.'))


#### Update matrix + psychometric — data vs fitted BE / SC


In [ ]:
# Simulate each model at its best-fit params, matching the data's session stimuli.
def sim_sessions(model, params):
    return [
        session_from_arrays(
            s['stimuli'],
            simulate_choices(model.lower(), params, s['stimuli'], s['categories'],
                             burn_in=GS_BURN_IN, seed=BASE_SEED),
            s['categories'], animal_id=DIAG_ANIMAL,
        )
        for s in diag['sessions']
    ]

be_sessions = sim_sessions('BE', be_best)
sc_sessions = sim_sessions('SC', sc_best)

emp_um, be_um, sc_um = compute_um(rec.sessions), compute_um(be_sessions), compute_um(sc_sessions)
emp_p,  be_p,  sc_p  = (compute_psychometric(rec.sessions),
                        compute_psychometric(be_sessions),
                        compute_psychometric(sc_sessions))

# Update matrices on a shared colour scale; MSE vs empirical in the titles.
vmax = max(np.nanmax(np.abs(u['um'])) for u in (emp_um, be_um, sc_um))
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
plot_um(emp_um, ax=ax[0], vmin=-vmax, vmax=vmax, colorbar=False, title=f'{DIAG_ANIMAL} empirical')
plot_um(be_um, ax=ax[1], vmin=-vmax, vmax=vmax, colorbar=False,
        title=f"BE fit (MSE={matrix_error(be_um['um'], emp_um['um']):.4f})")
plot_um(sc_um, ax=ax[2], vmin=-vmax, vmax=vmax, colorbar=True,
        title=f"SC fit (MSE={matrix_error(sc_um['um'], emp_um['um']):.4f})")
plt.tight_layout()
plt.show()

# Psychometric: data points + empirical / BE / SC curves overlaid.
fig, axp = plt.subplots(figsize=(5, 4))
plot_psychometric(emp_p, ax=axp, color='black', label='data', show_individual=False)
plot_psychometric(be_p, ax=axp, color=COLOURS['BE'], label='BE fit',
                  show_data=False, show_ci=False, show_individual=False)
plot_psychometric(sc_p, ax=axp, color=COLOURS['SC'], label='SC fit',
                  show_data=False, show_ci=False, show_individual=False)
axp.legend()
axp.set_title(f'{DIAG_ANIMAL}: psychometric — data vs fits')
plt.show()


## 2. Full Run

In [ ]:
from pathlib import Path

### 2a. Configuration

In [ ]:
# Point this at the gathered full-run results (copied back from the cluster).
folder_path = results_dir('grid_search', 'full', COHORT, FIT_TARGET)
print('full results <-', folder_path)


### 2b. Load the results

In [ ]:
res_full = load_cv_results(folder_path)
print(f'{res_full.comparison.shape[0]} animals loaded')


### 2c. Grid-Search Results

#### 2ci. Grid-search model identification

In [ ]:
show_model_id(res_full, label='Full:')


#### 2cii. GS parameter recovery


In [ ]:
show_recovery(res_full, label='Full:')
